# Lab 01: Baseline Agent

## Overview

In this notebook, we deploy an **intentionally unoptimized** customer support agent to establish baseline metrics. This baseline is the yardstick — every later lab measures its improvement against these numbers.

**What you'll learn:**
- How to deploy an agent to AgentCore Runtime
- How to configure Langfuse for observability
- How to invoke agents via the AgentCore API
- How to establish baseline metrics for cost and latency

## Prerequisites

- AWS account with Bedrock and AgentCore access
- Langfuse account (free tier works)
- `.env` file configured with your credentials

> ### ⚠️ These notebooks are for learning, not production
>
> The labs optimize one thing at a time so each change is easy to see, and they take shortcuts a real deployment shouldn't: resources are created imperatively from a notebook (not version-controlled **Infrastructure as Code** like CDK/CloudFormation/Terraform), there's a single environment with no **dev/staging/prod parity**, secrets live in a local `.env`, and the eval in Lab 07 runs by hand rather than as a **gated CI/CD pipeline**. For production, you'd also want automated rollback, load/cost testing at scale, alerting on the metrics we eyeball here, and human review of agent behavior. Treat the *techniques* as production-ready; treat the *plumbing* as a demo.

## Workshop Journey

```
[01 Baseline] → 02 Quick Wins → 03 Caching → 04 Routing → 05 Guardrails → 06 Skills + Gateway → 07 Evaluations
     ↑
  You are here
```

## Architecture

![Baseline Agent Architecture](images/00_agent_architecture.png)

This is a **single-agent loop**: one agent, one model, a set of tools. A user question comes in; the agent invokes the LLM (Claude on **Amazon Bedrock**), which reasons over the request and either answers directly or picks a tool to call. It has four tools — `get_return_policy`, `get_product_info`, `web_search`, and `get_technical_support` (which retrieves from a **Bedrock Knowledge Base**). The agent runs that reason → call tool → observe result → repeat cycle until it has an answer, then responds. No sub-agents, no hand-offs — just one loop. Every step is exported as **OpenTelemetry** traces to **Langfuse** for observability.

**A note on the deployment target:** the agent is hosted on **AgentCore Runtime**, but that choice isn't the point of the workshop — it's just a convenient managed, serverless host. The optimizations here (prompt design, caching, routing, guardrails, tool loading) are about the *agent itself* and apply no matter where it runs. If you deployed the same agent on **ECS** or **EKS** instead, the loop wouldn't change — you'd just take on more plumbing yourself: a web framework to expose an endpoint (FastAPI/Flask in Python, Express/Fastify in TypeScript, Gin/Echo in Go), a container and task/pod definitions, a load balancer, autoscaling, and your own OpenTelemetry wiring. AgentCore folds those in so we can focus on the agent.

**A note on Langfuse:** it's an open-source LLM observability platform — it collects each run as a trace so we can see tokens, cost, latency, and quality. Like the deployment target, the specific tool isn't important: it's just how we benchmark versions and inspect traces. Because the agent emits standard **OpenTelemetry**, you could swap in Arize Phoenix, LangSmith, Datadog, Grafana, or AWS's own CloudWatch GenAI Observability and get the same picture. We use Langfuse because it's open-source, free to self-host, and stores eval scores alongside the traces (handy for Lab 07). More on the trace data model just before we wire it in.

## The two pieces under the hood

This agent is built from two AWS-adjacent tools. If you ran the optional [intro notebook](00-intro-to-strands-agent.ipynb) these will be familiar; if not, here's the short version.

**[Strands Agents](https://strandsagents.com/)** — the open-source SDK that defines the agent. An agent is just three things: a **model** (Claude on Bedrock), a **system prompt** (its instructions), and **tools** (Python functions it can call). Strands runs the loop — the model reads the request and decides on its own whether to answer directly or call a tool, then loops until it's done. You'll see that loop as nested spans in the trace later.

**[Amazon Bedrock AgentCore Runtime](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/runtime.html)** — the managed, serverless host we deploy the agent to. It packages the code into a container, stands up the AWS resources (IAM, ECR), and gives us an API endpoint to invoke. It's framework-agnostic (Strands, LangGraph, CrewAI, …), so the same deploy flow works regardless of how the agent is written.

The agent's tools and prompt live in `agents/v1_baseline.py` — we look at it next.

## Step 1: Setup and Dependencies

Before starting, ensure you've run `uv sync` in the terminal and selected the `.venv` kernel in VS Code.

In [1]:
from __future__ import annotations

import json
import os
import uuid
from pathlib import Path

from dotenv import load_dotenv

load_dotenv(override=True)

import boto3
from bedrock_agentcore_starter_toolkit import Runtime

region = os.environ.get("AWS_DEFAULT_REGION", "us-east-1")
control_client = boto3.client("bedrock-agentcore-control", region_name=region)
data_client = boto3.client("bedrock-agentcore", region_name=region)
agentcore_runtime = Runtime()

print(f"Region: {region}")
print(f"Langfuse Host: {os.environ.get('LANGFUSE_BASE_URL', 'Not set')}")

/Users/tracilim/Projects/aws-bedrock-prompt-optimization-workshop/.venv/lib/python3.11/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.0.1)/charset_normalizer (3.4.5) doesn't match a supported version!
  warnings.warn(


Region: us-east-1
Langfuse Host: https://d1fnzg75c19u2d.cloudfront.net


## Step 2: Review the Baseline Agent

Our baseline agent (`agents/v1_baseline.py`) is intentionally unoptimized:

- **Verbose system prompt** (~1500 tokens instead of ~250)
- **No max_tokens limit** (model can generate unlimited output)
- **No prompt caching** (system prompt processed every request)
- **No stop sequences** (model decides when to stop)

Let's examine the key parts of the baseline agent:

In [2]:
# Read and display the baseline agent code
agent_file = Path("agents/v1_baseline.py")
print(agent_file.read_text())

"""
V1 Baseline Agent - Intentionally unoptimized for comparison.
No caching, verbose prompt, no max_tokens limit.
"""

from __future__ import annotations

import os

from bedrock_agentcore.runtime import BedrockAgentCoreApp
from strands import Agent
from strands.models import BedrockModel
from strands.telemetry import StrandsTelemetry

from utils.agent_config import MODEL_SONNET, setup_langfuse_telemetry
from utils.tools import get_product_info, get_return_policy, get_technical_support, web_search

setup_langfuse_telemetry()

app = BedrockAgentCoreApp()

# Verbose system prompt - intentionally unoptimized with common prompting mistakes:
# - Dense paragraphs without structure or hierarchy
# - Hedging language ("try to", "hopefully", "if possible", "maybe")
# - Filler phrases ("Please", "Can you please", "It would be great if")
# - No output format specification
# - Redundant adjective chains
# - No few-shot examples
SYSTEM_PROMPT = """
You are a customer support assistant and your job 

## Step 3: Configure the Agent for Deployment

Now we'll configure the agent for deployment to AgentCore Runtime.

In [3]:
agent_name = "customer_support_v1_baseline"
agent_file = str(Path("agents/v1_baseline.py").absolute())
requirements_file = str(Path("requirements-for-agentcore.txt").absolute())

print(f"Agent name: {agent_name}")
print(f"Agent file: {agent_file}")
print(f"Requirements: {requirements_file}")

Agent name: customer_support_v1_baseline
Agent file: /Users/tracilim/Projects/aws-bedrock-prompt-optimization-workshop/02-developer-journey/agents/v1_baseline.py
Requirements: /Users/tracilim/Projects/aws-bedrock-prompt-optimization-workshop/02-developer-journey/requirements-for-agentcore.txt


In [4]:
# Clean up any existing build files from previous labs
for f in ["Dockerfile", ".dockerignore", ".bedrock_agentcore.yaml"]:
    p = Path(f)
    if p.exists():
        p.unlink()
        print(f"Removed existing: {f}")

# Configure the runtime
print(f"Configuring agent: {agent_name}")
agentcore_runtime.configure(
    entrypoint=agent_file,
    auto_create_execution_role=True,
    auto_create_ecr=True,
    requirements_file=requirements_file,
    region=region,
    agent_name=agent_name,
)

Entrypoint parsed: file=/Users/tracilim/Projects/aws-bedrock-prompt-optimization-workshop/02-developer-journey/agents/v1_baseline.py, bedrock_agentcore_name=v1_baseline
Memory disabled - agent will be stateless
Configuring BedrockAgentCore agent: customer_support_v1_baseline


Removed existing: Dockerfile
Removed existing: .dockerignore
Removed existing: .bedrock_agentcore.yaml
Configuring agent: customer_support_v1_baseline


💡 No container engine found (Docker/Finch/Podman not installed)

✓ Default deployment uses CodeBuild (no container engine needed), For local builds, install Docker, Finch, or 
Podman

Memory disabled
Network mode: PUBLIC


📄 Generated Dockerfile: 
/Users/tracilim/Projects/aws-bedrock-prompt-optimization-workshop/02-developer-journey/Dockerfile

Generated .dockerignore: /Users/tracilim/Projects/aws-bedrock-prompt-optimization-workshop/02-developer-journey/.dockerignore
Setting 'customer_support_v1_baseline' as default agent
Bedrock AgentCore configured: /Users/tracilim/Projects/aws-bedrock-prompt-optimization-workshop/02-developer-journey/.bedrock_agentcore.yaml


ConfigureResult(config_path=PosixPath('/Users/tracilim/Projects/aws-bedrock-prompt-optimization-workshop/02-developer-journey/.bedrock_agentcore.yaml'), dockerfile_path=PosixPath('/Users/tracilim/Projects/aws-bedrock-prompt-optimization-workshop/02-developer-journey/Dockerfile'), dockerignore_path=PosixPath('/Users/tracilim/Projects/aws-bedrock-prompt-optimization-workshop/02-developer-journey/.dockerignore'), runtime='None', runtime_type=None, region='us-east-1', account_id='739907928487', execution_role=None, ecr_repository=None, auto_create_ecr=True, s3_path=None, auto_create_s3=False, memory_id=None, network_mode='PUBLIC', network_subnets=None, network_security_groups=None, network_vpc_id=None)

## A word on Langfuse, before we wire it in

Everything we optimize in this workshop — cost, latency, tokens, quality — we read from **[Langfuse](https://langfuse.com/docs)**, an open-source LLM observability platform. ([Basics Lab 03](../01-basics/03-langfuse-observability.ipynb) covers it in depth; this is the gist.)

The one concept to know is the **[trace](https://langfuse.com/docs/tracing)**: the full record of a single agent run. Each trace nests **observations** — spans for units of work, and *generations* for model calls, where Langfuse auto-computes [token usage and cost](https://langfuse.com/docs/model-usage-and-cost). It also stores [**scores**](https://langfuse.com/docs/scores/overview) (numeric/boolean evals attached to a trace) — that's how Lab 07 records answer quality next to the cost numbers.

We send traces by pointing the agent's OpenTelemetry exporter at Langfuse. The next step (modifying the Dockerfile) is what makes that happen — disabling AgentCore's default instrumentation so our Langfuse setup takes over.

## Step 4: Modify Dockerfile for Langfuse

We need to disable the default OpenTelemetry instrumentation and use Langfuse instead.

In [5]:
dockerfile_path = Path("Dockerfile")
if dockerfile_path.exists():
    content = dockerfile_path.read_text()
    # Replace opentelemetry-instrument wrapper with direct python call
    # Keep the correct module path: agents.v1_baseline
    if "opentelemetry-instrument" in content:
        import re

        content = re.sub(
            r'CMD \["opentelemetry-instrument", "python", "-m", "([^"]+)"\]', r'CMD ["python", "-m", "\1"]', content
        )
        dockerfile_path.write_text(content)
        print("Dockerfile modified for Langfuse")
    else:
        print("Dockerfile already configured or using different format")
else:
    print("Dockerfile not found - will be created during deployment")

Dockerfile modified for Langfuse


## Step 5: Deploy to AgentCore Runtime

Deploy the agent with Langfuse environment variables.

In [6]:
env_vars = {
    "LANGFUSE_BASE_URL": os.environ.get("LANGFUSE_BASE_URL"),
    "LANGFUSE_PUBLIC_KEY": os.environ.get("LANGFUSE_PUBLIC_KEY"),
    "LANGFUSE_SECRET_KEY": os.environ.get("LANGFUSE_SECRET_KEY"),
    "PYTHONUNBUFFERED": "1",
}

print("Deploying to AgentCore Runtime...")
print("This may take 5-10 minutes for the first deployment.")
launch_result = agentcore_runtime.launch(env_vars=env_vars, auto_update_on_conflict=True)
print(f"Agent deployed: {launch_result.agent_arn}")

🚀 Launching Bedrock AgentCore (cloud mode - RECOMMENDED)...
   • Deploy Python code directly to runtime
   • No Docker required (DEFAULT behavior)
   • Production-ready deployment

💡 Deployment options:
   • runtime.launch()                → Cloud (current)
   • runtime.launch(local=True)      → Local development
Memory disabled - skipping memory creation
Starting CodeBuild ARM64 deployment for agent 'customer_support_v1_baseline' to account 739907928487 (us-east-1)
Generated image tag: 20260602-042153-452
Setting up AWS resources (ECR repository, execution roles)...
Getting or creating ECR repository for agent: customer_support_v1_baseline


Deploying to AgentCore Runtime...
This may take 5-10 minutes for the first deployment.


ECR repository available: 739907928487.dkr.ecr.us-east-1.amazonaws.com/bedrock-agentcore-customer_support_v1_baseline
Getting or creating execution role for agent: customer_support_v1_baseline
Using AWS region: us-east-1, account ID: 739907928487
Role name: AmazonBedrockAgentCoreSDKRuntime-us-east-1-d683549b2a


✅ Reusing existing ECR repository: 739907928487.dkr.ecr.us-east-1.amazonaws.com/bedrock-agentcore-customer_support_v1_baseline


✅ Reusing existing execution role: arn:aws:iam::739907928487:role/AmazonBedrockAgentCoreSDKRuntime-us-east-1-d683549b2a
Execution role available: arn:aws:iam::739907928487:role/AmazonBedrockAgentCoreSDKRuntime-us-east-1-d683549b2a
Preparing CodeBuild project and uploading source...
Getting or creating CodeBuild execution role for agent: customer_support_v1_baseline
Role name: AmazonBedrockAgentCoreSDKCodeBuild-us-east-1-d683549b2a
Reusing existing CodeBuild execution role: arn:aws:iam::739907928487:role/AmazonBedrockAgentCoreSDKCodeBuild-us-east-1-d683549b2a
Using dockerignore.template with 47 patterns for zip filtering
Uploaded source to S3: customer_support_v1_baseline/source.zip
Updated CodeBuild project: bedrock-agentcore-customer_support_v1_baseline-builder
Starting CodeBuild build (this may take several minutes)...
Starting CodeBuild monitoring...
🔄 QUEUED started (total: 0s)
✅ QUEUED completed in 1.3s
🔄 PROVISIONING started (total: 2s)
✅ PROVISIONING completed in 6.3s
🔄 DOWNLOAD

Agent deployed: arn:aws:bedrock-agentcore:us-east-1:739907928487:runtime/customer_support_v1_baseline-q4CFH7BNIH


In [7]:
# Save the agent ARN for later use
agent_arn = launch_result.agent_arn
print(f"Agent ARN: {agent_arn}")

# Grant the auto-created execution role the shared runtime permissions
# (SSM KB lookup + Bedrock Retrieve + ApplyGuardrail) used across all labs.
from utils.runtime_helpers import ensure_runtime_permissions

ensure_runtime_permissions(region)

Agent ARN: arn:aws:bedrock-agentcore:us-east-1:739907928487:runtime/customer_support_v1_baseline-q4CFH7BNIH
Granted customer-support runtime permissions to AmazonBedrockAgentCoreSDKRuntime-us-east-1-d683549b2a


## Step 6: Test the Baseline Agent

Let's invoke the agent with some test queries to establish our baseline metrics.

In [8]:
def invoke_agent(prompt):
    """Invoke the agent via AgentCore API."""
    response = data_client.invoke_agent_runtime(
        agentRuntimeArn=agent_arn,
        runtimeSessionId=str(uuid.uuid4()),
        payload=json.dumps({"prompt": prompt}).encode(),
    )
    return json.loads(response["response"].read().decode("utf-8"))

In [9]:
# Import Langfuse metrics helper
from utils.langfuse_metrics import (
    clear_metrics,
    collect_metric,
    get_latest_trace_metrics,
    print_metrics,
    print_metrics_table,
)

# Clear any previously collected metrics
clear_metrics()


def invoke_agent_with_metrics(prompt, test_name=""):
    """Invoke the agent and fetch + print Langfuse metrics."""
    response = invoke_agent(prompt)
    print(response)

    # Fetch and print metrics from Langfuse
    metrics = get_latest_trace_metrics(
        agent_name="customer-support-v1-baseline",
        wait_seconds=5,
        max_retries=5,
        timeout_seconds=120,
    )
    print_metrics(metrics, test_name)

    # Collect metrics for summary table
    collect_metric(metrics, test_name)

    return response, metrics

In [10]:
# Standard test prompts - same across all notebooks for consistent comparison
TEST_PROMPTS = [
    # Single tool: get_return_policy
    ("Return Policy", "What is your return policy for laptops?"),
    # Single tool: get_product_info
    ("Product Info", "Tell me about your smartphone options"),
    # Single tool: get_technical_support (Bedrock KB)
    ("Technical Support", "My laptop won't turn on, can you help me troubleshoot?"),
    # Multi-tool: get_product_info + get_return_policy
    ("Multi-part Question", "I want to buy a laptop. What are the specs and what's the return policy?"),
    # No tool: General greeting
    ("General Question", "Hello! What can you help me with today?"),
]

# Run all tests and collect metrics
for test_name, prompt in TEST_PROMPTS:
    print("=" * 60)
    print(f"Test: {test_name}")
    print("=" * 60)
    response, metrics = invoke_agent_with_metrics(prompt, test_name=test_name)

Test: Return Policy
Great news — here's a full breakdown of our **laptop return policy** at TechMart Electronics:

- 📅 **Return Window:** You have **30 days** from the date of purchase to return a laptop.
- 📦 **Condition Requirements:** The laptop must be in its **original packaging** with **all accessories included** and **no physical damage**.
- 🔄 **Return Process:** You can initiate a return through our **online RMA portal** or bring it back **in-store**.
- 💰 **Refund Timeline:** Once your return is received and inspected, your refund will be processed within **7–10 business days**.
- 🚚 **Return Shipping:**
  - **Free** return shipping if the laptop has a **defect**.
  - **Customer pays** for return shipping for a **change of mind** return.
- 🔁 **Restocking Fee:**
  - **No restocking fee** for defective returns.
  - A **15% restocking fee** applies for change of mind returns.
- 🛡️ **Warranty:** All laptops come with a **1-year manufacturer warranty**, and **extended warranty options

In [11]:
# Print summary table of all test metrics
print_metrics_table()

# Save metrics for comparison in later notebooks
from utils.langfuse_metrics import save_metrics

save_metrics("v1")


                                  METRICS SUMMARY
               Test Latency    Cost Input Output Cache Read Tokens Cache Write Tokens
      Return Policy   8.37s $0.0277 7,307    382                 0                  0
       Product Info   8.29s $0.0282 7,344    411                 0                  0
  Technical Support   6.78s $0.0153 3,554    312                 0                  0
Multi-part Question   9.68s $0.0323 7,641    623                 0                  0
   General Question   5.48s $0.0145 3,549    260                 0                  0
---------------------------------------------------------------------------------------------------------
  TOTALS: Latency(avg): 7.72s | Cost: $0.1180 | Input: 29,395 | Output: 1,988
          Cache Read Tokens: 0 | Cache Write Tokens: 0

Metrics saved as 'v1' → .lab_metrics.json


## Step 7: View the trace in Langfuse

Every invocation above was sent to Langfuse as a **trace** — the full record of one run: what came in, what the agent did, what came out, and what it cost. The rest of the workshop reads its numbers from here, so it's worth getting familiar with the view now.

**Open one:**

1. Go to your Langfuse app (URL printed below).
2. Click **Tracing** in the left sidebar.
3. Click any trace in the list to open it.

Run the next cell for your URL, then take a look — the screenshot after it walks through what you're seeing.

In [ ]:
langfuse_base_url = os.environ.get("LANGFUSE_BASE_URL", "https://cloud.langfuse.com")
print(f"Open Langfuse: {langfuse_base_url}")
print("Then: left sidebar → Tracing → click any trace.")
print("\nFilter the list by the tags 'baseline' / 'no-optimization' to find this lab's runs.")

### What a trace looks like

Here's one of this lab's traces — the "return policy for laptops" query:

<img src="images/trace%201.png" alt="Langfuse trace of the baseline agent" width="650">

A few things to read off it:

- **The tree (top-left).** The run nests into spans: the agent invocation → an event-loop cycle → the model `chat` call → the `get_return_policy` tool call. Each row shows its own latency and cost, so you can see *where* time and money go, not just the total.
- **The header numbers.** This single run took **9.52s**, cost **$0.044**, and used **10,718 input + 814 output tokens**. That input count is the baseline problem in one number — the ~1,500-token system prompt rides along on every request, with no caching to discount it.
- **Tags** (`baseline`, `no-optimization`) are what you filter the list by, and what later labs use to compare versions.
- **Input / Output (right).** The exact query in and the full answer out — handy for spotting a rambling or off-format response.

This is the lens for the whole workshop: every optimization from here on should move one of these numbers down **without** making the output worse. Keep this view open as you go.

## Summary

In this notebook, we:

1. Deployed an unoptimized baseline agent to AgentCore Runtime
2. Configured Langfuse for observability
3. Ran test scenarios to establish baseline metrics
4. Identified areas for optimization:
   - Verbose system prompt (~1500 tokens)
   - No output token limits
   - No prompt caching
   - No model routing

**Next Steps:** In the next notebook, we'll apply "quick wins" optimizations:
- Concise system prompt
- max_tokens limit
- stop_sequences

**Next notebook:** [02-quick-wins.ipynb](./02-quick-wins.ipynb)

## Cleanup (Optional)

Uncomment and run if you want to delete the agent.

In [13]:
# # Uncomment to delete resources created in this lab
# agentcore_runtime.destroy(delete_ecr_repo=True)
# print(f"Deleted agent and ECR repository: {agent_name}")